# Density functional theory

Companion notebook to Chapter 8 of *Quantum mechanics for many-particle
systems*.  Every number quoted in the chapter is produced here; the code is
the same as in `BookManybody/BookMaterial/Programs/dft.py`.

The theme is that the local density approximation is not a black box but
something one *builds*, from a system that can be solved exactly.  We build it
twice: once for the three-dimensional electron gas of Chapter 6, where the
answer is analytic, and once numerically for the one-dimensional trap of
Chapters 2 and 6, so that Kohn-Sham, Hartree-Fock and the exact answer can be
compared for one and the same Hamiltonian.

Contents:

1. Where the LDA comes from
2. The exchange hole and its sum rule
3. Building a functional for the model interaction
4. Solving the Kohn-Sham equations
5. The self-interaction error
6. How good is the local approximation?
7. Thomas-Fermi, and why orbitals came back

In [ ]:
import sys, os
import numpy as np
import matplotlib.pyplot as plt

sys.path.insert(0, os.path.join("..", "BookManybody", "BookMaterial", "Programs"))
import dft

np.set_printoptions(precision=6, suppress=True, linewidth=120)

## 1. Where the local density approximation comes from

$$E_{\rm xc}^{\rm LDA}[n] = \int d^3r\, n(\mathbf r)\,
  \epsilon_{\rm xc}\!\left(n(\mathbf r)\right),$$

with $\epsilon_{\rm xc}$ taken from the uniform gas.  For exchange the answer
is analytic,

$$\epsilon_x(n) = -\frac34\left(\frac{3}{\pi}\right)^{1/3}n^{1/3},$$

and evaluated on the uniform gas it must reproduce the $-0.916/r_s$ of
Chapter 6.  That it does is a tautology — the LDA is *defined* to be exact
there — but it is a useful check that the conventions line up.

In [ ]:
dft.demo_lda_from_the_gas()

In [ ]:
rs = np.linspace(0.5, 8.0, 400)
n = dft.ExchangeLDA3D.density_from_rs(rs)
fig, ax = plt.subplots(figsize=(7, 4.2))
ax.plot(rs, 2 * dft.ExchangeLDA3D.eps_x(n), label=r"LDA: $\epsilon_x(n(r_s))$")
ax.plot(rs, -0.916 / rs, "k--", lw=0.9, label=r"chapter 6: $-0.916/r_s$")
ax.set_xlabel(r"$r_s$"); ax.set_ylabel(r"$\epsilon_x$  [Ry]")
ax.set_title("the exchange functional against the electron-gas result")
ax.legend(); fig.tight_layout(); plt.show()

## 2. The exchange hole

Each electron moves inside a hole in the density of its neighbours.  For the
uniform gas the pair correlation function is exact at the Hartree-Fock level,

$$g(y) = 1 - \frac92\left[\frac{\sin y - y\cos y}{y^3}\right]^2,
\qquad y = k_F s,$$

with $g(0)=\tfrac12$: same-spin electrons never coincide, opposite-spin ones
are uncorrelated at this level.  The hole contains exactly one missing
electron, and that sum rule is a large part of why local functionals work at
all — the energy depends mostly on the *charge* of the hole, which is fixed
exactly, and only weakly on its shape.

In [ ]:
dft.demo_exchange_hole()

In [ ]:
y = np.linspace(1e-6, 20.0, 2000)
g = dft.exchange_hole(y)
fig, ax = plt.subplots(1, 2, figsize=(12, 4.2))
ax[0].plot(y, g)
ax[0].axhline(1.0, color="k", lw=0.7); ax[0].axhline(0.5, color="k", ls=":", lw=0.7)
ax[0].set_xlabel(r"$k_F s$"); ax[0].set_ylabel("$g(s)$")
ax[0].set_title("pair correlation function")
ax[1].plot(y, y**2 * (g - 1.0))
ax[1].axhline(0.0, color="k", lw=0.7)
ax[1].set_xlabel(r"$k_F s$"); ax[1].set_ylabel(r"$y^2\,[g(y)-1]$")
ax[1].set_title("the sum-rule integrand")
fig.tight_layout(); plt.show()

## 3. Building a functional for the model interaction

The trap of Chapters 2 and 6 uses a softened Coulomb interaction in one
dimension, for which no tabulated functional exists.  So we do what is done in
practice: solve the uniform gas once, tabulate, interpolate.  For a spinless
1D gas $k_F = \pi n$ and $\rho^{(1)}(s) = \sin(k_F s)/\pi s$, so

$$\epsilon_x(n) = -\frac1n\int_0^\infty ds\, v(s)
  \left[\frac{\sin(k_F s)}{\pi s}\right]^2 .$$

In [ ]:
dft.demo_lda_1d()

In [ ]:
lda = dft.ExchangeLDA1D()
n = np.linspace(0.01, 2.5, 400)
fig, ax = plt.subplots(figsize=(7, 4.2))
ax.plot(n, lda.eps_x(n), label=r"$\epsilon_x(n)$")
ax.plot(n, lda.v_x(n), label=r"$v_x(n) = d(n\epsilon_x)/dn$")
ax.plot(lda.grid[::12], lda.table[::12], "k.", ms=4,
        label="tabulated points")
ax.set_xlabel("$n$"); ax.set_title("the exchange functional we just built")
ax.legend(); fig.tight_layout(); plt.show()

## 4. Solving the Kohn-Sham equations

$$\left(-\frac{\hbar^2}{2m}\nabla^2 + v_{\rm KS}(\mathbf r)\right)
  \psi_i = \varepsilon_i\psi_i,
\qquad
v_{\rm KS} = v_{\rm ext} + v_{\rm H}[n] + v_{\rm xc}[n],$$

iterated to self-consistency exactly as in Chapter 6.  Same trap, same basis
of eight oscillator orbitals, same interaction — so Kohn-Sham, Hartree-Fock
and (for two particles) the exact answer are directly comparable.

In [ ]:
dft.demo_kohn_sham()

In [ ]:
lda = dft.ExchangeLDA1D()
fig, ax = plt.subplots(1, 2, figsize=(12, 4.4))

ks = dft.KohnSham1D(4, lda=lda); ks.run()
hf = dft.hartree_fock(4)
grid, phi = dft.oscillator_basis(8)
n_free = (phi[:4] ** 2).sum(axis=0)
n_hf = ((phi.T @ hf["C"][:, :4]) ** 2).sum(axis=1)

ax[0].plot(grid, ks.n, label="Kohn-Sham")
ax[0].plot(grid, n_hf, "--", label="Hartree-Fock")
ax[0].plot(grid, n_free, ":", label="non-interacting")
ax[0].set_xlim(-5, 5); ax[0].set_xlabel("$x$"); ax[0].set_ylabel("$n(x)$")
ax[0].set_title("densities, $N = 4$"); ax[0].legend()

it, energies, drifts = zip(*ks.history)
ax[1].semilogy(it, drifts)
ax[1].set_xlabel("iteration"); ax[1].set_ylabel(r"max $|\Delta n|$")
ax[1].set_title("self-consistency")
fig.tight_layout(); plt.show()

## 5. The self-interaction error

For one particle there is nothing to interact with, so the exact energy is the
oscillator ground state, $0.5$.  Hartree-Fock gets this identically right —
the direct and exchange terms cancel term by term.  A local functional cannot,
because it sees only the density, and the density of one electron is
indistinguishable from the density of half of two.

In [ ]:
dft.demo_self_interaction()

The residue is what makes plain LDA overbind anions, underestimate reaction
barriers and give band gaps that are too small: an electron is spuriously
repelled by itself and spreads out to reduce that repulsion.  Removing it is
the business of self-interaction corrections and of hybrid functionals, which
mix back a fraction of exact exchange.

## 6. How good is the local approximation?

We can isolate the error of *locality* alone by evaluating, on one and the
same density, both the exact exchange energy of the determinant and the local
estimate.

In [ ]:
dft.demo_local_approximation()

In [ ]:
Ns = [1, 2, 3, 4]
ratios = []
for N in Ns:
    ks = dft.KohnSham1D(N, lda=lda); ks.run()
    vbar = dft.trap_two_body(8)
    occ = ks.C[:, :N]
    both = 0.5 * np.einsum("ai,bj,abcd,ci,dj->", occ, occ, vbar, occ, occ)
    exact_x = both - ks.hartree_energy(ks.n)
    ratios.append(ks.exchange_energy(ks.n) / exact_x)

fig, ax = plt.subplots(figsize=(6.5, 4))
ax.plot(Ns, ratios, "o-")
ax.axhline(1.0, color="k", lw=0.8)
ax.set_xticks(Ns); ax.set_xlabel("$N$")
ax.set_ylabel(r"$E_x^{\rm local} / E_x^{\rm exact}$")
ax.set_title("the local approximation improves as the density smooths out")
fig.tight_layout(); plt.show()

## 7. Thomas-Fermi, and why orbitals came back

Thomas-Fermi applies the local approximation to the kinetic energy as well,
which removes orbitals entirely and leaves an algebraic equation for $n(x)$.
It is enormously cheaper — and it fails, not mainly in magnitude but in
structure: the density it produces has no shells at all, and Teller proved
that within the model no molecule is bound.  The Kohn-Sham construction exists
precisely to avoid this, by keeping orbitals for the one term that a local
approximation handles badly.

In [ ]:
dft.demo_thomas_fermi()

In [ ]:
grid, phi = dft.oscillator_basis(8)
dx = grid[1] - grid[0]
Ns = np.array([1, 2, 3, 4, 6, 8])
ratio = []
for N in Ns:
    n = (phi[:N] ** 2).sum(axis=0)
    exact = sum(k + 0.5 for k in range(N)) - 0.5 * float(n @ grid**2) * dx
    ratio.append(float(dft.ThomasFermi.kinetic_density(n).sum()) * dx / exact)

fig, ax = plt.subplots(1, 2, figsize=(12, 4.2))
ax[0].plot(Ns, ratio, "o-")
ax[0].axhline(1.0, color="k", lw=0.8)
ax[0].set_xlabel("$N$"); ax[0].set_ylabel(r"$T_{\rm TF} / T_{\rm exact}$")
ax[0].set_title("Thomas-Fermi kinetic energy")

n8 = (phi[:8] ** 2).sum(axis=0)
ax[1].plot(grid, n8, label="exact (orbitals)")
ax[1].set_xlim(-6, 6); ax[1].set_xlabel("$x$"); ax[1].set_ylabel("$n(x)$")
ax[1].set_title("the shell structure Thomas-Fermi cannot produce")
ax[1].legend()
fig.tight_layout(); plt.show()

## The full program

Everything above lives in `BookManybody/BookMaterial/Programs/dft.py`, which
runs as a script and prints all seven demonstrations of the chapter.

In [ ]:
print(open(dft.__file__).read())